# Anukulan: plant memory and a case stack, live

An indigenous LP / MILP / QP solver, written from scratch for SIH26119 (MRPL). This notebook builds it from source and reproduces two measurements from the idea deck on a refinery planning model:

1. **Plant memory**: the plan re-solved day after day as prices and demand move, each day started from the answer the day before ended on.
2. **A case stack**: the base plan and 60 what-if cases around it (crude prices, crude supply, product demand, unit outages), each started from the base case's answer.

Every case is solved twice on the identical model, from scratch and from memory, and the two answers are compared. The warm answer is also checked against the model's own rows, so memory can save work but never change an answer.

Runs on a free CPU runtime in about three minutes. No GPU needed. **Runtime > Run all** to start.

In [ ]:
# Build from source: a C++20 compiler and CMake, nothing else.
!git clone --depth 1 https://github.com/team-vertexx/anukulan.git
%cd anukulan
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j 4 2>&1 | tail -n 1

## 1. Every component tested

Fourteen test suites: sparse algebra, LU, simplex, presolve, crossover, branch and bound, cuts, the first-order method and its backends.

In [ ]:
!ctest --test-dir build 2>&1 | tail -n 3

## 2. One refinery plan, solved and checked

A multi-period planning model in real units (barrels, rupees, ppm): crude purchase, distillation, hydrotreating, cracking, blending to sulphur specifications, inventory. `vs original` is the check on the original, unscaled model: the largest row violation and the largest bound violation of the answer.

In [ ]:
!build/anukulan simplex data/refinery/refinery.mps

## 3. Plant memory: ten days of the plan

Crude prices move about 1% a day, product prices about 0.7%, demand within 4%. Each day is solved cold and from the previous day's basis. `same` means the two answers agree.

In [ ]:
!python3 scripts/refinery_family.py --days 10
!build/anukulan family data/refinery/family/day_*.mps

## 4. A case stack: the base plan and 60 what-ifs

One crude's price moved by 3% or 6% either way, one crude's supply halved, one product's demand moved by 10%, or the CDU, FCC or hydrotreater cut back in one period. Every case starts from the base case's basis. The summary at the bottom is the number on the slide.

In [ ]:
!python3 scripts/refinery_cases.py
!build/anukulan family --from-first data/refinery/cases/case_*.mps | tail -n 12

## What you just saw

- Starting from the plant's last certified answer cuts the simplex work by an order of magnitude, and **not one answer changes**.
- The same memory serves a whole stack of what-ifs around one decision.

The full tables (12, 24 and 52 periods), the GPU measurements on a Tesla T4, and the command behind every number are in [`docs/RESULTS.md`](https://github.com/team-vertexx/anukulan/blob/master/docs/RESULTS.md).